# 4.3 Playing with ROS2

Time to play with realistic sensors! 🤖 In notebook 4.2 we built our own, very simple, laser scanner simulator. Now we
are going to work with the sensors of a robot simulated by
[MVSim](https://mvsimulator.readthedocs.io/en/latest/index.html), through **ROS2**: what
they publish, what their messages contain, where they are mounted on the robot and how noisy they are, connecting all
of it with the sensor models of notebooks 4.1 and 4.2.

By the end of this notebook you will be able to configure the sensors of a simulated robot, tell what each one
publishes, read the content of a laser scan, measure its noise, and find out where each sensor is mounted.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/turtlebot3_world_mvsim_scenario.png" alt="" width="750">
  <figcaption>Fig. 1. The TurtleBot3 Burger robot in the TurtleBot3 World of MVSim, together with the beams of its 2D laser scanner (in red).</figcaption>
</figure>

## 4.3.1 How to work with this notebook

**This notebook is not meant to be run.** ROS2 is a set of programs (*nodes*) running at the same time on your own
machine, so this notebook plays the role of a **lab guide and report**: keep it open while you work on your terminals,
and write your answers back into it:

1. **Commands**: paste the command you used (and, when asked, its output) into the markdown code blocks below.
2. **Screenshots**: a capture of the simulator, of RViz2 or of a terminal.

**Assumed setup.** The same as in notebook 2.1: Ubuntu 24.04, ROS2 Jazzy and MVSim, installed as a ROS2 package
(`ros-jazzy-mvsim`).

*Tip: you will modify an MVSim world file several times. Make a backup copy of it before your first change (e.g.
`sudo cp demo_turtlebot_world.world.xml demo_turtlebot_world.world.xml.bak`), so you can always go back to a working
version.*

## 4.3.2 The TurtleBot3 world

In this notebook we are going to work with the **`demo_turtlebot_world`** world of MVSim. Its robot is a
[TurtleBot3 Burger](https://emanual.robotis.com/docs/en/platform/turtlebot3/overview/), a small differential drive
robot by ROBOTIS widely used for teaching and research with ROS, and the world reproduces the *TurtleBot3 World*, the
arena with columns where it is usually tested.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/turtlebot3_burger_sim_vs_real.png" alt="" width="700">
  <figcaption>Fig. 2. The TurtleBot3 Burger in MVSim (left) and the real robot (right). In both, the 2D laser scanner is on top.</figcaption>
</figure>

The robot is defined in the vehicle class `definitions/turtlebot3_burger.vehicle.xml`, which mounts three sensors when
its `default_sensors` argument is true: a 360° **2D laser scanner** (simulated as a
[RPLIDAR A2](https://www.slamtec.com/en/Lidar/A2)), an **RGB camera** and an **IMU**. In this notebook we are going
to focus on the laser scanner, although we will also keep the IMU. Besides, the camera is a Full HD one (1920 × 1080
pixels) running at 30 images per second, which the simulator renders through the GPU: on a computer without a decent GPU, or inside a
virtual machine, it can make the simulation crawl. So, before starting, we are going to remove it.

### **<span style="color:green"><b><i>ASSIGNMENT 1: Preparing the world</i></b></span>**

As with any vehicle class, `turtlebot3_burger.vehicle.xml` is shared by every world of the system installation, so we
are not going to touch it. Instead, we will switch off the default sensors of the class from the world file, and mount
our own sensors on the robot there.

**What to do?**

1. Find where the world file `demo_turtlebot_world.world.xml` was installed in your system, and open it with your
   favourite editor. Since it belongs to the system installation, you will need `sudo` to save your changes. *Hint: it
   lives in the `mvsim_tutorial` directory of the installed `mvsim` package, and `ros2 pkg prefix` tells you where a
   package was installed.* Write below the commands you used.

```bash
# TODO: your command here
```


2. Find the `<include>` of the `turtlebot3_burger` vehicle class and set its `default_sensors` argument to `false`, so
   the robot arrives with no sensors at all. Paste below the piece of XML you changed.

```xml
<!-- TODO: your answer here -->
```


3. Now mount the sensors on the robot from the world file. Inside the `<vehicle name="r1">` tag, right after its
   `<init_pose>`, add the following block (no need to type it, just copy it). The laser scanner and the IMU are the same
   as in the vehicle class, and so are their names, so the RViz2 configuration of the demo keeps working (only its
   camera panel will stay empty).

```xml
<!-- 2D laser scanner (RPLIDAR A2), as in the TurtleBot3 class -->
<include file="../definitions/rplidar-a2.sensor.xml"
  raytrace_3d="true"
  sensor_x="-0.032" sensor_z="0.171" sensor_yaw="0"
  sensor_hz="10.0"
  sensor_name="laser1"
/>

<!-- IMU, as in the TurtleBot3 class -->
<include file="../definitions/imu.sensor.xml"
  sensor_x="0.0" sensor_y="0.05" sensor_z="0.145"
  sensor_period_sec="$f{1/200.0}"
  sensor_name="imu1"
/>
```

   Paste below the resulting `<vehicle>` block:

```xml
<!-- TODO: your answer here -->
```


4. Launch the simulation, and include a **screenshot** showing both the MVSim GUI and RViz2:

```bash
ros2 launch mvsim demo_turtlebot_world.launch.py
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 4.3.3 The robot's sensors in ROS2

Each sensor simulated by MVSim publishes its readings in its own topic, with a message type suited to the kind of data
it produces. Besides the data themselves, every sensor message starts with a **header** (`std_msgs/msg/Header`) that
carries two pieces of information that are key in robotics:

- `stamp`: **when** the measurement was taken.
- `frame_id`: **where** it was taken, that is, the name of the reference frame in which the data are expressed (for a
  sensor, usually a frame attached to it). We will come back to this in section 4.3.6.

### **<span style="color:green"><b><i>ASSIGNMENT 2: Sensor inventory</i></b></span>**

With the simulation running, find out in another terminal what each sensor of the robot publishes.

**What to do?**

1. Fill in the table below. *Hints: `ros2 topic list -t` gives you the topics together with their types;
   `ros2 topic echo <topic> --once --no-arr` prints a single message hiding its long arrays, so you can read its
   `frame_id` at a glance; and remember that there is a `ros2 topic` subcommand that measures the rate of a topic.* The
   first row is filled in for you, as an example.

| Sensor | Topic | Message type | `frame_id` | Rate (Hz) |
|---|---|---|---|---|
| 2D laser scanner | `/laser1` | `sensor_msgs/msg/LaserScan` | `laser1` | ~10 |
| IMU |  |  |  |  |
| Wheel encoders (odometry) |  |  |  |  |

```bash
# TODO: your command here
```


2. Compare the rates you measured with the configuration of each sensor in the world file. Do they match? *Note: if
   your computer cannot keep the simulation in real time, MVSim prints a warning ("Simulation slower than real time")
   and the rates you measure will be somewhat lower.*

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 4.3.4 The laser scanner

A 2D laser scanner publishes `sensor_msgs/msg/LaserScan` messages. Each message is a whole **scan**, that is, a set of
$K$ measurements $\mathbf{z} = [\mathbf{z}_1, \dots, \mathbf{z}_K]$, each one $\mathbf{z}_k = [r_k, \alpha_k]^T$ in
polar coordinates relative to the sensor, just like the observations of the `Laser2D` class of notebook 4.2.

However, the message does **not** store $K$ pairs $[r_k, \alpha_k]^T$. The beams of a scanner are always emitted at the
same, evenly spaced, angles: the first one at an angle `angle_min`, and each of the following ones `angle_increment`
radians after the previous one. So, instead of the $K$ angles, the message stores just these two values,
and then **only the ranges**, in the `ranges` list, sorted by angle. The angle of each range is given by
its position $k$ in that list (counting from $k = 0$):

$$
\alpha_k = \texttt{angle\_min} + k \cdot \texttt{angle\_increment}
$$

### **<span style="color:green"><b><i>ASSIGNMENT 3: Anatomy of a laser scan</i></b></span>**

**What to do?**

1. Find the command that shows the definition of the `sensor_msgs/msg/LaserScan` message type, that is, its fields
   together with the comments that explain them. *Hint: message types are **interfaces**, and there is a ROS2
   subcommand for them.*

```bash
# TODO: your command here
```


2. Print a single message of `/laser1` without its long arrays, and fill in the table below, which relates its fields
   with the parameters of the `Laser2D` class of notebook 4.2. Give the angles in degrees.

```bash
# TODO: your command here
```

| `Laser2D` parameter (4.2) | `LaserScan` field(s) | Value |
|---|---|---|
| Field of view (`FOV`) |  |  |
| Angular resolution (`resolution`) |  |  |
| Max distance (`max_distance`) |  |  |
| Number of beams |  |  |


3. Using the expression of $\alpha_k$ given above, which index $k$ of `ranges` corresponds, approximately, to the beam
   pointing straight ahead of the robot? And which directions do the first and the last readings point
   to?

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 4.3.5 Sensor noise and configuration

As we modeled in notebooks 4.1 and 4.2, the measurements of a real sensor are not exact, and MVSim adds noise to the
simulated ones too. For the laser scanner, `definitions/rplidar-a2.sensor.xml` defines a zero-mean Gaussian noise in
the range of each beam, with standard deviation `range_std_noise` (set with the `sensor_std_noise` argument), and
another one in its angle, with standard deviation `angle_std_noise_deg`.

### **<span style="color:green"><b><i>ASSIGNMENT 4: Measuring the noise of the scanner</i></b></span>**

Keep the robot **standing still** (do not publish anything in `/cmd_vel`).

**What to do?**

1. Print the `ranges` of `/laser1` a few times and compare the value of the **same beam** in different scans. Write
   below the command you used, and the values you got. *Hint: `ros2 topic echo /laser1 --once --field ranges` prints
   only that field.*

```bash
# TODO: your command here
```


2. How much do those values vary? Is it consistent with the `range_std_noise` of the scanner?

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>ASSIGNMENT 5: Configuring the scanner</i></b></span>**

Now let's change the scanner. Edit the world file again, and add these two arguments to the `<include>` of `laser1`:

```xml
sensor_std_noise="0.10"
max_range="3.0"
```

**What to do?**

1. Paste below the resulting `<include>` block.

```xml
<!-- TODO: your answer here -->
```


2. Launch the simulation again and include a **screenshot** of RViz2 showing the scan with the new configuration.
   Compare it with the original one: what has changed, and why?

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


3. Print a single message of `/laser1`, first without its arrays and then only its `ranges`. Which value does
   `range_max` take now? And which value do the beams that do not hit anything within the maximum range take?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


4. Restore the original configuration of the scanner, removing those two arguments, before going on.

### <font color="blue"><b><i>Thinking about it</i></b></font>

Having explored the laser scanner, **answer the following questions**:

- The `Laser2D` class of notebook 4.2 returned each reading as a pair $[r_k, \alpha_k]^T$. What does a `LaserScan`
  message gain by storing only the ranges?

  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Which problem would a mapping algorithm have if it treated the readings equal to `range_max` as real obstacles?

  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- What do we gain, and what do we lose, by increasing the number of beams of a scanner?

  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 4.3.6 Where are the sensors? Reference frames and tf

The ranges of a `LaserScan` are expressed in the frame of the scanner (its `frame_id`, `laser1`), not in the frame of
the robot, nor in the world one. To use them, for instance to place the scanned points on a map as we did in notebook
4.2, we need the pose of the scanner on the robot and the pose of the robot in the world, and compose them: a point
$\mathbf{z}_c$ measured in cartesian coordinates in the scanner frame is expressed in the world frame as
$\mathbf{x} \oplus \mathbf{x}_s \oplus \mathbf{z}_c$, being $\mathbf{x}$ the pose of the robot and $\mathbf{x}_s$ the
pose of the sensor on the robot (see Fig. 3). In notebook 4.2 we took the shortcut of placing the scanner at the center
of the robot, that is, $\mathbf{x}_s = [0, 0, 0]^T$.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/tf-frames-and-tree.gif" alt="" width="800">
  <figcaption>Fig. 3. Left, the reference frames of a mobile robot with a sensor, seen from above while the robot moves (with their x axes in red and their y axes in green, as RViz2 draws them): <code>base_link</code> and the frame of the sensor move with the robot, while <code>map</code> and <code>odom</code> stay fixed in the world, and the point measured by the sensor is always expressed in the world frame as the same point. Right, the tf tree of these frames.</figcaption>
</figure>

ROS2 manages all these poses with [**tf2**](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Tf2.html). Its
main ideas are:

- **Frames and transforms.** A *frame* is a reference system with a name, like `base_link` or `laser1`. A *transform*
  between a parent frame and a child frame is the pose of the child expressed in the parent: a translation and a
  rotation. It is the same idea as the poses $[x, y, \theta]^T$ of chapter 3, extended to 3D (ROS stores the rotations
  as quaternions, but its tools also show them as roll, pitch and yaw angles).
- **The tf tree.** Every frame has a single parent, so the frames form a tree, as in the right part of Fig. 3, and
  there is a single path between any two frames. To get the transform between two of them, tf2 composes the transforms
  along that path, which is exactly the composition of poses $\oplus$. For example, to get the pose of the sensor in
  the `odom` frame, tf2 follows the path `odom` → `base_link` → `sensor` of Fig. 3 and composes its two transforms: the
  pose of the robot in `odom`, $\mathbf{x}$, and the pose of the sensor on the robot, $\mathbf{x}_s$. The result is
  $\mathbf{x} \oplus \mathbf{x}_s$.
- **Who publishes the transforms.** The nodes publish the transforms they know in the `/tf` topic (those that change
  over time, e.g. the pose of the robot) and in `/tf_static` (the fixed ones, e.g. the pose of a sensor on the robot,
  although some nodes publish them in `/tf` too). Every transform carries a timestamp, so tf2 can give the transform
  at the instant a measurement was taken, given by the `stamp` of its header (section 4.3.3).
- **Standard frames.** Mobile robots in ROS follow the conventions of [REP 105](https://www.ros.org/reps/rep-0105.html):
  - `map`: a frame fixed in the world. The pose of the robot in it does not drift, but it can jump when a
    localization algorithm corrects it.
  - `odom`: another frame fixed in the world, in which the odometry expresses the pose of the robot. This pose is
    continuous (no jumps), but it drifts over time. A localization node publishes the transform `map` → `odom` to
    correct that drift. In our simulation there is no such node, so MVSim
    publishes it as the identity.
  - `base_link`: a frame rigidly attached to the robot. The transform `odom` → `base_link` is the pose of the robot
    $\mathbf{x}$ given by the odometry.
  - `base_footprint`: the projection of `base_link` on the floor.
  - The frames of the sensors (e.g. `laser1`), which hang from the frames of the robot.

Fig. 4 shows the frames of our robot: on the left, the frames of its two sensors drawn by MVSim on the TurtleBot3
Burger (option *View sensor poses* of its GUI), and on the right, all its frames as RViz2 draws them with its `TF`
display.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/turtlebot3_burger_mvsim_rviz_frames_v2.png" alt="" width="700">
  <figcaption>Fig. 4. Reference frames of the TurtleBot3 Burger: those of its sensors in MVSim (left), and the whole set in RViz2 (right). In the right image, some labels overlap: at the bottom, <code>base_link</code> and <code>base_footprint</code> (its projection on the floor, which in this robot coincides with it), and on the right, <code>map</code> and <code>odom</code>, which coincide in our simulation.</figcaption>
</figure>

### **<span style="color:green"><b><i>ASSIGNMENT 6: The tf tree of the robot</i></b></span>**

**What to do?**

1. With the simulation running, generate a diagram of the tf tree. *Hint: the `tf2_tools` package has an executable
   called `view_frames`, which listens to `/tf` for a few seconds and saves the tree in a PDF file in the current
   directory.* Write the command below and include a **screenshot** of the tree.

```bash
# TODO: your command here
```


   Compare it with Fig. 3. Which frames does the tree of the TurtleBot3 have? Which of them are frames of sensors, and
   which is their parent? Which frame of the tree does not appear in the figure?

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


2. Print the transformation from `base_link` to `laser1`. *Hint: the `tf2_ros` package has an executable called
   `tf2_echo`, which takes the two frames as arguments.* Does it match the `<include>` of the scanner in the world
   file?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


3. Now print the transformation from `odom` to `base_link`, while you drive the robot from another terminal, publishing
   in `/cmd_vel`. What does it represent?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


4. Finally, let's check that tf2 composes poses just as we do. Stop the robot and print the transformation from `odom`
   to `base_link`: its translation $x, y$ and its yaw angle $\theta$ (the last value of `Rotation: in RPY (radian)`)
   give you the pose of the robot on the plane, $\mathbf{x} = [x, y, \theta]^T$. From step 2 you already know the pose
   of the scanner on the robot, $\mathbf{x}_s = [x_s, y_s, \theta_s]^T$. Compute by hand the pose of the scanner in
   `odom`, that is:

   $$
   \mathbf{x} \oplus \mathbf{x}_s =
   \begin{bmatrix}
   x + x_s \cos\theta - y_s \sin\theta \\
   y + x_s \sin\theta + y_s \cos\theta \\
   \theta + \theta_s
   \end{bmatrix}
   $$

   whose height is just the sum of the heights $z$ of both transformations. Then print the transformation from `odom`
   to `laser1`: does it match your result?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>OPTIONAL: Observing a landmark with tf</i></b></span>**

tf2 is not only for the frames of the robot: any node can publish a frame, for example one for a landmark of the map.
Then, tf2 can tell us where that landmark is with respect to the scanner, that is, the observation that the scanner
should provide, which is the predicted observation $\hat{\mathbf{z}} = \mathbf{h}(\mathbf{x}, \mathbf{m})$ of notebook
4.1.

As landmark, let's take the center of the white column at $(-1.1, 0)$ m in the world. Note that tf2 does not know the
world frame of MVSim: the odometry starts at zero where the robot starts, which the world file places at
$(-2, -0.5)$ m with a yaw of 0°. So, in the `odom` frame (and in `map`, which coincides with it here), that column is at
$(0.9, 0.5)$ m.

1. Launch the simulation again, so the robot is at its initial pose, and, without moving it, publish a static frame
   called `landmark` at that position from a new terminal:

```bash
ros2 run tf2_ros static_transform_publisher --x 0.9 --y 0.5 --frame-id odom --child-frame-id landmark
```

   Then print the transformation from `laser1` to `landmark`, and, from its translation, compute the range and bearing
   $[r, \alpha]^T$ of the landmark as seen from the scanner.

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>


2. Which index $k$ of `ranges` points to the landmark? Print the `ranges` of `/laser1` and compare the reading of that
   beam with $r$. Why are they not equal?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>OPTIONAL: Recording and replaying sensor data</i></b></span>**

Real robots record their sensor data so they can be processed later, offline, as many times as needed: that is how
most robotics datasets are built. ROS2 does it with
[**rosbags**](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools/Recording-And-Playing-Back-Data/Recording-And-Playing-Back-Data.html).

1. While you drive the robot around, record the topics of the laser scanner, the IMU and the odometry, together with
   `/tf` and `/tf_static`, using `ros2 bag record`. Then stop the recording and show its content with `ros2 bag info`.
2. Close the simulation, play the bag back with `ros2 bag play`, and check that those topics are being published
   again, even though the simulator is no longer running.

Write below the commands you used and the output of `ros2 bag info`.

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I asked AI to help with (bullets):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### A.2 AI review
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this mobile robotics lab notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant ROS2 concepts and tools used, not particular commands or topics),
- one on how to improve the work (most critical aspect: limitations, parameter choices, robustness, possible mistakes).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

Finally, **do you agree with them?**, **would you have selected others?**
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>